# Who Goes There: train one config on Kaggle

Trains one config of `config.yaml` (the small CNN, ResNet-18 arm A or arm B) on the 512 px CCT20
cache with `trailcam.train.train`, then prints the ledger row and the checkpoint's SHA-256 (item
6.02; the recipe is `docs/plan.md` sections 5 and 6). The code, `config.yaml` and the ledger all
come from one clone of the public repo at the commit `COMMIT`, so the config hash in the row is
the hash of that commit's config. This notebook never scores cis_test or trans_test: that is the
final scoring (items 9.01 and 9.02), with the user's go-ahead.

## How to run on Kaggle

1. Import this notebook into Kaggle: File > Import Notebook, with its GitHub link
   (https://github.com/codx999/who-goes-there/blob/main/notebooks/kaggle_train.ipynb).
2. Add Input: attach the private 512 px cache dataset of item 6.03 (`annotations/`, `images/` and
   `day_night.json`). Kaggle mounts it at `/kaggle/input/datasets/<user>/<slug>/<folder>`; the
   default `DATA_ROOT` is the user's dataset (`codx888`, `cct20-512`, `cct20_512`), so another
   account or dataset name changes it. If `DATA_ROOT` is wrong, section 3 lists the folders under
   `/kaggle/input` that hold a `day_night.json`.
3. Keep exactly one parameter cell, the first code cell: delete any copy of it left by an earlier
   import or edit, since a later cell's assignment silently overrides the first. Edit only
   `COMMIT` (and `DATA_ROOT` for another dataset): set `COMMIT` to the full sha of the `main`
   commit the coordinator gives, at or after the merge of item 6.07. The notebook cannot name its
   own commit, so the default is the `main` commit it was written against, which predates 6.07
   and is refused in section 2. `CONFIG_NAME` (`small_cnn` for 6.04) and `RUN_SEED` (`None`
   keeps the config's seed) change only when a run asks for another config or seed.
4. Session options: Accelerator GPU T4 x2, the choice of `docs/plan.md` section 5 (not GPU
   P100: Kaggle's torch has no kernels for it), and Internet on (the clone, pip and the
   ImageNet weights). The session has two T4s and training uses the first one; the GPU
   quota is 30 hours a week.
5. Rehearse: run sections 1 to 3 cell by cell (parameters, clone and install, the code, GPU and
   cache checks; a few minutes), then stop the session so no GPU time is idle. Section 3 refuses
   a cache with any image missing, with counts only: a session start can load part of the
   dataset, so this is checked, not assumed.
6. Save Version > Save & Run All (Commit). The version runs in a fresh session and repeats every
   check before training. The checkpoint (`/kaggle/working/models/`) and the clone with its
   ledger are kept as the version's output.
7. Paste the printed ledger row into `docs/ledger.md` (items 6.04 and 6.05) and note the commit
   sha in the progress-log entry. If the training cell stops on a LEAKAGE ALARM, the row and
   the alarms are printed above it: stop and report them before any other run (R16).

Commit this notebook with its outputs cleared (Edit > Clear All Outputs); it prints counts only,
never images or metadata records.

Data: Caltech Camera Traps (CCT20), licence
[CDLA-Permissive 1.0](https://cdla.dev/permissive-1-0/). Please cite: Sara Beery, Grant Van
Horn, Pietro Perona. "Recognition in Terra Incognita." Proceedings of the European Conference
on Computer Vision (ECCV), 2018. https://arxiv.org/abs/1807.04975. Code: MIT.

In [ ]:
# Parameters: the only parameter cell (How to run, step 3); set COMMIT, then Save & Run All.
from pathlib import Path

CONFIG_NAME = "small_cnn"  # small_cnn, resnet18_arm_a or resnet18_arm_b (config.yaml)
RUN_SEED = None  # None: the config's run seed (0); 1 or 2 for the extra seeds (plan.md section 6)
COMMIT = "e321151aed8aaf0cedc4581685235898796a511d"  # the main commit the coordinator gives
DATA_ROOT = Path("/kaggle/input/datasets/codx888/cct20-512/cct20_512")  # the 6.03 dataset

REPO_URL = "https://github.com/codx999/who-goes-there"  # the public repo
WORK = Path("/kaggle/working")  # kept as the version's output
CLONE = WORK / "who-goes-there"
OUT_DIR = WORK / "models"  # the checkpoint

## 1. Check the parameters and pin one GPU

Fails in seconds, before anything is cloned or installed, and prints the parameters in
effect (a second parameter cell would show here). `CUDA_VISIBLE_DEVICES=0` is set
here, before torch is first imported (by the `trailcam` import of section 3), so torch sees
the first T4 of GPU T4 x2 only and every run trains on one T4.

In [ ]:
import os
import re
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4 of the two; torch is not imported yet
print(f"Python {sys.version.split()[0]}")
if not re.fullmatch(r"[0-9a-f]{40}", COMMIT):
    raise ValueError(f"COMMIT must be a full 40-character commit sha, got {COMMIT!r}")
if CONFIG_NAME not in ("small_cnn", "resnet18_arm_a", "resnet18_arm_b"):
    raise ValueError(f"CONFIG_NAME must be a training config of config.yaml, got {CONFIG_NAME!r}")
print(f"CONFIG_NAME {CONFIG_NAME}, RUN_SEED {RUN_SEED}, COMMIT {COMMIT}, DATA_ROOT {DATA_ROOT}")

## 2. Clone the repo at `COMMIT` and install `trailcam` from the clone

`config.yaml` and `docs/ledger.md` are not in the wheel, so `train.py` is given both from this
clone (`--config`, `--ledger`): code, config and config hash come from one commit. This is the
project's one `pip install` (the notebook exception in `CLAUDE.md`). The package's dependency
floors lie below Kaggle's versions, so pip keeps the preinstalled torch, torchvision, Pillow,
timm and PyYAML; the cell checks that it did. A `COMMIT` older than item 6.07 lacks the
cache check of section 3 and is refused before the install.

In [ ]:
import subprocess

if CLONE.exists():  # a second run in one interactive session reuses the clone
    print(f"reusing the clone at {CLONE}")
else:
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(CLONE)], check=True)
    subprocess.run(["git", "-C", str(CLONE), "checkout", "--quiet", "--detach", COMMIT], check=True)
head = subprocess.run(
    ["git", "-C", str(CLONE), "rev-parse", "HEAD"], check=True, capture_output=True, text=True
).stdout.strip()
changed = subprocess.run(
    ["git", "-C", str(CLONE), "status", "--porcelain", "--untracked-files=no"],
    check=True,
    capture_output=True,
    text=True,
).stdout.split("\n")
changed = [line for line in changed if line and not line.endswith(" docs/ledger.md")]
if head != COMMIT or changed:
    raise RuntimeError(
        f"{CLONE} is at {head} with {len(changed)} changed files, not a clean {COMMIT}: "
        "restart the session (Run > Factory reset) and run again"
    )
if not (CLONE / "trailcam" / "data" / "cache.py").is_file():  # added by item 6.07
    raise RuntimeError(
        f"commit {head} predates item 6.07 (no trailcam/data/cache.py), which this notebook "
        "needs: set COMMIT to the main commit the coordinator gives, at or after its merge"
    )
print(f"commit {head}")

In [ ]:
from importlib.metadata import version

KEPT = ("torch", "torchvision", "Pillow", "timm", "PyYAML")  # Kaggle's preinstalled builds
preinstalled = {name: version(name) for name in KEPT}
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", str(CLONE)], check=True)
after_install = {name: version(name) for name in KEPT}
if after_install != preinstalled:
    raise RuntimeError(f"pip replaced preinstalled packages: {preinstalled} -> {after_install}")
print(f"trailcam {version('trailcam')} from commit {head}; kept {after_install}")

## 3. Check the installed code, the GPU and the cache

`trailcam` must not be imported yet in this kernel (Python would keep an earlier commit's
modules), and every module of the installed package must match the clone byte for byte.

`check_gpu` refuses a session without a GPU, with other than the one pinned device visible, or
whose device torch was not built for (a P100, sm_60, under Kaggle's torch 2.11), so such a
session fails here and not at the first CUDA call of training. The CPU count is printed next
to the config's DataLoader workers, with a warning if the workers outnumber the CPUs
(`num_workers` is in the config hash, so a change is the user's decision).

The cache must hold the annotations, the images and the day/night sidecar. If `DATA_ROOT`
lacks one, the folders holding a `day_night.json` at any depth under `/kaggle/input` (up to
`MAX_DEPTH` levels) are listed. The metadata is loaded once and shown as counts only (images per
split, images dropped per rule), then `check_images` refuses the cache unless the image of every
kept record of the five splits is under `images/`, naming the expected and found counts, never a
file name or an id (R03). `train.py` runs the same check again before the split is drawn.

In [ ]:
if "trailcam" in sys.modules:  # imported earlier in this kernel: maybe another commit's code
    raise RuntimeError(
        "trailcam is already imported in this kernel and would not be reloaded from the clone: "
        "restart the session (Run > Factory reset) and run again"
    )
import trailcam  # noqa: E402
from trailcam.data.cache import check_images, sidecar_folders  # noqa: E402
from trailcam.data.metadata import DAY_NIGHT_SIDECAR, load_metadata  # noqa: E402
from trailcam.train import train  # noqa: E402
from trailcam.train.config import load_config  # noqa: E402
from trailcam.train.device import check_gpu, workers_warning  # noqa: E402
from trailcam.train.train import CACHE_ANNOTATIONS, CACHE_IMAGES, file_sha256  # noqa: E402

installed = Path(trailcam.__file__).parent
modules = sorted(p.relative_to(CLONE / "trailcam") for p in (CLONE / "trailcam").rglob("*.py"))
differ = [
    str(m) for m in modules if file_sha256(installed / m) != file_sha256(CLONE / "trailcam" / m)
]
if differ:
    raise RuntimeError(f"the installed trailcam differs from the clone in {differ}")
print(f"{len(modules)} trailcam modules at {installed} match commit {head}")

In [ ]:
print(check_gpu())
num_workers = load_config(CONFIG_NAME, CLONE / "config.yaml").num_workers
print(f"{os.cpu_count()} CPUs; config {CONFIG_NAME} uses {num_workers} DataLoader workers")
warning = workers_warning(num_workers, os.cpu_count())
if warning:
    print(warning, file=sys.stderr)

In [ ]:
needed = {
    "annotations": DATA_ROOT / CACHE_ANNOTATIONS,
    "images": DATA_ROOT / CACHE_IMAGES,
    "day/night sidecar": DATA_ROOT / DAY_NIGHT_SIDECAR,
}
missing = [name for name, path in needed.items() if not path.exists()]
if missing:
    roots = [str(folder) for folder in sidecar_folders(Path("/kaggle/input"))]
    raise FileNotFoundError(
        f"DATA_ROOT {DATA_ROOT} lacks the {', '.join(missing)}; folders under /kaggle/input "
        f"holding a {DAY_NIGHT_SIDECAR}: {roots or 'none (is the dataset attached?)'}"
    )
meta = load_metadata(needed["annotations"], needed["day/night sidecar"])
print(meta)
print(f"{sum(p.is_file() for p in needed['images'].rglob('*'))} files under {CACHE_IMAGES}/")
print(check_images(meta, needed["images"]))

## 4. Train

`trailcam.train.train.main`, the same entry point as `python -m trailcam.train.train`. It checks
the out dir and the ledger before training and the cache's images again, seeds `random`, NumPy and torch, trains with early
stopping on val macro-F1, restores and saves the best epoch, scores trans_val once (reported,
never selected on), prints the row and any leakage alarm, then appends the row to the clone's
ledger. It never reads cis_test or trans_test. A leakage alarm makes this cell fail after the
row is printed (R16).

In [ ]:
LEDGER = CLONE / "docs" / "ledger.md"
ledger_before = LEDGER.read_text(encoding="utf-8").splitlines()
argv = ["--config-name", CONFIG_NAME, "--config", str(CLONE / "config.yaml")]
argv += ["--data-root", str(DATA_ROOT), "--out-dir", str(OUT_DIR), "--ledger", str(LEDGER)]
if RUN_SEED is not None:
    argv += ["--run-seed", str(RUN_SEED)]
print("python -m trailcam.train.train " + " ".join(argv))
try:
    exit_code = train.main(argv)
except SystemExit as stop:  # argparse refused an argument or a path, before training
    raise RuntimeError(f"train.py stopped before training (exit {stop.code}), see above") from None
if exit_code == train.ALARM_EXIT:
    raise RuntimeError(
        "LEAKAGE ALARM (R16): the row and the alarms are printed above and the row is in the "
        "clone's ledger; stop and report them before any other run"
    )
if exit_code != 0:
    raise RuntimeError(f"train.py exited {exit_code}")

## 5. The ledger row

The one row the run appended to the clone's ledger, checked against `LEDGER_COLUMNS`, and the
checkpoint's SHA-256 computed again from the file kept under `OUT_DIR`. Paste the row into
`docs/ledger.md`.

In [ ]:
ledger_after = LEDGER.read_text(encoding="utf-8").splitlines()
new_rows = ledger_after[len(ledger_before) :]
if ledger_after[: len(ledger_before)] != ledger_before or len(new_rows) != 1:
    raise RuntimeError(f"expected one new row at the end of {LEDGER}, found {len(new_rows)}")
row = new_rows[0]
cells = [cell.strip() for cell in row.strip().strip("|").split("|")]
values = dict(zip(train.LEDGER_COLUMNS, cells, strict=True))
if train.ledger_row(values) != row:
    raise RuntimeError("the new ledger line does not read back as a row of LEDGER_COLUMNS")
checkpoint = OUT_DIR / f"{values['run id']}.pt"
digest = file_sha256(checkpoint)
if digest != values["checkpoint sha256"]:
    raise RuntimeError(f"{checkpoint} hashes to {digest}, the row says another hash")
print(f"commit {head}, config {CONFIG_NAME}, run seed {values['run seed']}")
print(f"checkpoint {checkpoint}, sha256 {digest}")
print("ledger row, for docs/ledger.md:")
print(train.ledger_header())
print(row)